# Fast Video Embeddings

This notebook provides a method for more quickly generating multi-modal embeddings of videos using Google's `multimodalembedding@001` embedding model. See more details [here](https://cloud.google.com/vertex-ai/generative-ai/docs/embeddings/get-multimodal-embeddings).

## What you'll need

* A Google Cloud Account and Google Cloud Project

## Basic Setup
### Install dependencies

In [ ]:
%pip install \
    google-cloud-alloydb-connector[asyncpg]==1.4.0 \
    sqlalchemy==2.0.36 \
    pandas==2.2.2 \
    vertexai==1.70.0 \
    asyncio==3.4.3 \
    greenlet==3.1.1 \
    langchain_google_alloydb_pg==0.9.3 \
    datasets==3.4.0 \
    --quiet

### Authenticate to Google Cloud within Colab
If you're running this on google colab notebook, you will need to Authenticate as an IAM user.

In [ ]:
from google.colab import auth

auth.authenticate_user()

### Connect Your Google Cloud Project

In [ ]:
# @markdown Please fill in the value below with your GCP project ID and then run the cell.

# Please fill in these values.
project_id = "your-project-id"  # @param {type:"string"}

# Quick input validations.
assert project_id, "⚠️ Please provide a Google Cloud project ID"

# Configure gcloud.
!gcloud config set project {project_id}

### Enable APIs for AlloyDB and Vertex AI

You will need to enable these APIs in order to create an AlloyDB database and utilize Vertex AI as an embeddings service!

In [ ]:
!gcloud services enable alloydb.googleapis.com aiplatform.googleapis.com

### Configure Logging

In [ ]:
import logging
import sys

# Configure the root logger to output messages with INFO level or above
logging.basicConfig(level=logging.INFO, stream=sys.stdout, format='%(asctime)s[%(levelname)5s][%(name)14s] - %(message)s',  datefmt='%H:%M:%S', force=True)

## Connect to AlloyDB
You will need a Postgres AlloyDB instance for the following stages of this notebook. Please set the following variables to connect to your instance.

In [ ]:
# @markdown Please fill in the both the Google Cloud region and name of your AlloyDB instance. Once filled in, run the cell.

# Please fill in these values.
region = "us-central1"  # @param {type:"string"}
cluster = "your-cluster"  # @param {type:"string"}
instance = "your-instance"  # @param {type:"string"}
database = "your-database"  # @param {type:"string"}
table = "video_embeddings"   # @param {type:"string"}
bucket = "your-bucket"  # @param {type:"string"}
password = input("Please provide a password to be used for 'postgres' database user: ")


## Download Sample Videos

In [ ]:
# Create bucket if none is provided
import uuid
from google.cloud import storage

client = storage.Client()
random_suffix = uuid.uuid4().hex[:6]  # Get a 6-character hexadecimal suffix
bucket_name = f"test-videos-{random_suffix}"

if not bucket:
    bucket = client.create_bucket(bucket_name)
    print(f"Bucket {bucket.name} created.")
else:
    print(f"Using provided bucket: {bucket}")
    bucket = client.bucket(bucket)

In [ ]:
# Login to Huggingface
!huggingface-cli login

In [ ]:
# Download Huggingface repo snapshot with video files
from huggingface_hub import snapshot_download
local_path = snapshot_download(repo_id="yaak-ai/lerobot-driving-school", repo_type="dataset")
print(f"Repo saved to: {local_path}")

In [ ]:
# Upload test videos to GCS so that the embedding model can access them.
import os
import uuid
from google.cloud import storage

"""Recursively finds and uploads video files to a GCS bucket."""
client = storage.Client()

video_dir = os.path.join(local_path, "videos", "chunk-000")

gcs_uris = []

if not os.path.exists(video_dir):
    raise(f"Directory {video_dir} does not exist.")

for root, _, files in os.walk(video_dir):
    for file in files:
        if file.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):  # Add more video extensions if needed
            local_file_path = os.path.join(root, file)
            relative_path = os.path.relpath(local_file_path, video_dir) #get the relative path to keep the directory structure in GCS.
            blob_path = relative_path.replace(os.sep, '/') #replace windows backslash with forward slash for GCS.
            blob = bucket.blob(blob_path)

            blob.upload_from_filename(local_file_path)
            gcs_uri = f"gs://{bucket.name}/{blob.name}"
            gcs_uris.append(gcs_uri)
            print(f"Uploaded {local_file_path} to {gcs_uri}")

print(f"Done uploading {len(gcs_uris)} files.")


## Run the embeddings workflow


In [ ]:
import asyncio
from typing import Union, List, AsyncIterator, Any
import vertexai
from vertexai.vision_models import MultiModalEmbeddingModel, Video
from vertexai.vision_models import VideoSegmentConfig
from google.api_core.exceptions import ResourceExhausted


async def embed_video(
    video_uri: str,
    model: MultiModalEmbeddingModel,
    retries: int = 100,
    delay: int = 30,
) -> List[Any]:
    
    logger = logging.getLogger("embed_objects")

    # Retry loop
    for attempt in range(retries):
        try:
            # Get embeddings 
            embeddings = model.get_embeddings(
                video=Video.load_from_file(video_uri),
                video_segment_config=VideoSegmentConfig(interval_sec=10),
            )
            #print(f"Generated embeddings: {embeddings}")
            return embeddings.video_embeddings
            #return {'id': str(uuid.uuid4()), 'contents': video_uri, 'embeddings': embeddings.video_embeddings, 'metadata': ''}

        except Exception as e:
            if attempt < retries - 1:  # Retry only if attempts are left
                logger.warning(f"Error: {e}. Retrying in {delay} seconds...")
                await asyncio.sleep(delay)  # Wait before retrying
            else:
                logger.error(f"Failed to get embeddings for video: {video_uri} after {retries} attempts.")
    return []


async def embed_videos_concurrently(
    video_uris: List[str],
    model: MultiModalEmbeddingModel,
    max_concurrency: int = 5,
) -> AsyncIterator[List[dict[str, Union[str, List[float]]]]]:

    logger = logging.getLogger("embed_videos")
    queue = asyncio.Queue()
    for uri in video_uris:
        await queue.put(uri)

    tasks = {}  # Use a dictionary to store tasks and their URIs
    while not queue.empty() or tasks:
        # Wait for at least one task to complete *if* tasks exist
        if tasks:
            done, _ = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)
            for task in done:
                video_uri = tasks.pop(task) # Get the URI *and* remove the task.
                result = await task  # Await the task to get the result
                if result:
                    logger.info(f"Embedding task completed: Processed video {video_uri}.")
                    yield (video_uri,result)

        # Calculate how many new tasks to add
        num_to_add = min(max_concurrency - len(tasks), queue.qsize())

        # Create tasks up to the concurrency limit, AFTER handling completed tasks
        for _ in range(num_to_add):
            video_uri = await queue.get()
            task = asyncio.create_task(embed_video(video_uri, model))
            print(f"Adding task for {video_uri}...")
            tasks[task] = video_uri  # Store the URI *with* the task


In [ ]:
import time
import vertexai
from vertexai.vision_models import MultiModalEmbeddingModel, Video
from vertexai.vision_models import VideoSegmentConfig
from datetime import datetime, timezone

vertexai.init(project=project_id, location="us-central1")


# Model to use for generating embeddings
model_name = "multimodalembedding@001"

### Embeddings workflow ###


async def run_embeddings_workflow(
    video_uris: List[str] = gcs_uris,
    embed_video_concurrency: int = 4
):
    # Establish to AlloyDB engine client
    from langchain_google_alloydb_pg import AlloyDBEngine
    from google.cloud.alloydb.connector import IPTypes

    alloydb_engine = await AlloyDBEngine.afrom_instance(
        project_id=project_id,
        region=region,
        cluster=cluster,
        instance=instance,
        database=database,
        user='postgres',
        password=password,
        ip_type=IPTypes.PRIVATE,
    )
    # [END pinecone_vectorstore_alloydb_migration_get_client]
    print("Langchain AlloyDB client initiated.")

    # Initialize Vector Store Table in AlloyDB
    from langchain_google_alloydb_pg import Column

    await alloydb_engine.ainit_vectorstore_table(
            table_name=table,
            vector_size=1408,
            overwrite_existing=True,
            # Customize the ID column types with `id_column` if not using the UUID data type
            id_column=Column("id", "uuid")  #  Default is Column("langchain_id", "UUID")
        )
    print("AlloyDB vector store table initiated.")
    
    # Initialise VertexAI and the model to be used to generate embeddings
    vertexai.init(project=project_id, location=region)
    model = MultiModalEmbeddingModel.from_pretrained(model_name)

    print(f"Vertex AI model {model_name} initialized.")

    # Connect instance of AlloyDBVectorStore to AlloyDB Vector Store Table
    from langchain_google_alloydb_pg import AlloyDBVectorStore

    vs = await AlloyDBVectorStore.create(
            engine=alloydb_engine,
            embedding_service=model,
            table_name=table,
            id_column="id"  # Must match id_column defined in ainit_vectorstore_table()
        )
    print("Connected to AlloyDB vector store.")

    start_time = time.monotonic()
    
    # Embed videos and update the database concurrently
    async for current_video_uri, embeddings_batch in embed_videos_concurrently(
        video_uris,
        model,
        max_concurrency=embed_video_concurrency
    ):
        if embeddings_batch:  # Ensure we have embeddings
            #Prepare embeddings and texts to be added to the database
            ids = []
            texts = []
            embeddings = []
            metadatas = []

            # The video embeddings are a list of embeddings, one per segment.
            for index, embedding in enumerate(embeddings_batch):
                # Format: uri:start_offset:end_offset
                texts.append(f"{current_video_uri}:{embedding.start_offset_sec}:{embedding.end_offset_sec}")
                embeddings.append(embedding.embedding)
                # Add some metadata.  
                metadatas.append({"date_created": str(datetime.now(timezone.utc)), 
                                  "video_uri": current_video_uri,
                                  "start_offset_sec": embedding.start_offset_sec,
                                  "end_offset_sec": embedding.end_offset_sec
                                  }) 
                ids.append(str(uuid.uuid4())) #generate an id

            #Add to database
            logging.info(f"Adding embeddings to database. Number of embeddings: {len(embeddings)}") # useful for checking
            await vs.aadd_embeddings(
                ids=ids,
                texts=texts,
                embeddings=embeddings,
                metadatas=metadatas,
            )


    end_time = time.monotonic()
    elapsed_time = end_time - start_time

    # Release database connections and close the connector

    print(f"Job started at: {time.ctime(start_time)}")
    print(f"Job ended at: {time.ctime(end_time)}")
    print(f"Total run time: {elapsed_time:.2f} seconds")


await run_embeddings_workflow()